# 01 — Basic Damage Simulation

Run a single scenario (Warrior vs NPC) and inspect the damage distribution.

In [ ]:
import logging
from pathlib import Path

from omega.model.constants import SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_ANATOMY, SKILLID_WRESTLING
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import summary_table, format_table_html
from omega.reporting.plots import damage_histogram, damage_breakdown
from omega.logging import setup_logging

# Suppress noisy stub warnings — only show errors in notebook output
setup_logging(level=logging.ERROR)

# Works whether CWD is the project root or the notebooks/ directory
SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

In [ ]:
scenario = Scenario(
    attacker=CombatantSpec(
        name="Warrior",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
        str_=100, dex_=100, int_=25,
        class_levels={"IsWarrior": 5},
        weapon=WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP),
    ),
    defender=CombatantSpec(
        name="Target Dummy",
        is_npc=True,
        str_=50, dex_=50, int_=50,
        hp=500,
        skills={SKILLID_WRESTLING: 60},  # NPC combat skill — affects hit chance
        armor=ArmorSpec(name="Plate", ar=30),
    ),
    iterations=200,
    base_seed=42,
)

result = run_scenario(scenario, shard=shard)
print(f"Successes: {result.success_count}/{result.iteration_count}")
print(f"Hit rate: {result.ratios.hit_rate:.1%}")
print(f"Mean damage (overall): {result.damage_stats.mean:.2f}")
print(f"Mean damage (on hit):  {result.damage_stats_on_hit.mean:.2f}")
print(f"Range: {result.damage_stats.min:.0f} – {result.damage_stats.max:.0f}")

In [ ]:
# Damage distribution histogram
damage_histogram(result, title="Warrior vs NPC — 200 hits")

In [ ]:
# Base / absorbed / final breakdown
damage_breakdown(result, title="Damage Breakdown")

In [ ]:
# HTML summary table — overall stats (including misses) and on-hit stats
from IPython.display import HTML
from omega.simulation.stats import SimulationResult

rows = summary_table(
    SimulationResult(cells=[result]),
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95", "hit_rate", "count"],
)
HTML(format_table_html(rows))

## Enchanted Weapon Comparison (V1.5)

Apply an enchantment using `enchant_with()` and compare against the plain weapon.

In [ ]:
from omega.config.enchantments import Enchantment
from omega.reporting.plots import comparison_overlay, enchantment_comparison
from omega.reporting.tables import comparison_table

# Create an enchanted version of the same weapon.
# ChanceOfEffect controls how often the spell fires (75 = 75%);
# EffectCircle controls spell power (circle 10 = highest tier).
plain_sword = WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP)
fire_sword = WeaponSpec(
    name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP,
    properties={"ChanceOfEffect": 75, "EffectCircle": 10},
).enchant_with(Enchantment.OF_DAEMONS_BREATH)  # Fireball on hit

print(f"Plain:     hitscript={plain_sword.hitscript}")
print(f"Enchanted: hitscript={fire_sword.hitscript}")
print(f"           properties={fire_sword.properties}")

In [ ]:
RUN_KW = dict(shard=shard)

attacker = CombatantSpec(
    name="Warrior",
    skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
    str_=100, dex_=100, int_=25,
    class_levels={"IsWarrior": 5},
)
defender = CombatantSpec(
    name="Target Dummy", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    skills={SKILLID_WRESTLING: 60},
    armor=ArmorSpec(name="Plate", ar=30),
)

import dataclasses

enchant_results = {}
for label, weapon in [("Plain", plain_sword), ("Daemon's Breath", fire_sword)]:
    atk = dataclasses.replace(attacker, weapon=weapon)
    enchant_results[label] = run_scenario(
        Scenario(attacker=atk, defender=defender, iterations=200, base_seed=42),
        **RUN_KW,
    )
    ds = enchant_results[label].damage_stats
    r = enchant_results[label].ratios
    print(f"  {label:20s}  mean={ds.mean:6.2f}  hit_rate={r.hit_rate:.1%}  "
          f"spell_strike={r.spell_strike_rate:.1%} (on hit: {r.spell_strike_rate_on_hit:.1%})")

In [ ]:
# Overlaid histograms: plain vs enchanted
comparison_overlay(enchant_results, title="Plain vs Daemon's Breath Enchantment")

In [ ]:
# Side-by-side stat comparison with enchantment-specific columns
rows = comparison_table(
    enchant_results,
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95",
           "hit_rate", "spell_strike_rate", "spell_strike_rate_on_hit"],
)
HTML(format_table_html(rows))

## DPS & Swing Timing (V2)

The simulator computes POL-conformant swing delay from weapon speed and attacker DEX, enabling damage-per-second (DPS) analysis.

In [ ]:
# DPS metrics are computed automatically from weapon speed + attacker DEX
ts = result.timing
if ts:
    print(f"Swing delay:   {ts.swing_delay_ms:.0f}ms")
    print(f"Swings/sec:    {ts.swings_per_second:.2f}")
    print(f"DPS (mean):    {ts.dps_mean:.1f}")
    print(f"DPS (on-hit):  {ts.dps_on_hit:.1f}")
    print(f"Effective DPS: {ts.effective_dps:.1f}")

# Table with DPS columns alongside damage stats
rows = summary_table(
    SimulationResult(cells=[result]),
    stats=["mean", "swing_delay_ms", "swings_per_sec", "dps_mean", "effective_dps", "hit_rate"],
)
HTML(format_table_html(rows))

## Astral Damage (V2)

Astral weapons use a completely different damage pipeline: they drain **mana and stamina** instead of HP, scaling with Spirit Speak and EvalInt instead of STR and Tactics.

In [ ]:
from omega.model.constants import SKILLID_SPIRITSPEAK, SKILLID_EVALINT, SKILLID_MEDITATION

# Astral mage attacker — Spirit Speak and EvalInt are the key scaling skills
astral_scenario = Scenario(
    attacker=CombatantSpec(
        name="Astral Mage",
        skills={
            SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 80,
            SKILLID_SPIRITSPEAK: 100, SKILLID_EVALINT: 100,
        },
        str_=50, dex_=100, int_=100,
        class_levels={"IsMage": 5},
        weapon=WeaponSpec(name="Astral Blade", damage="3d6+2",
                         attribute=SKILLID_SWORDSMANSHIP,
                         properties={"Astral": 1}),
    ),
    defender=CombatantSpec(
        name="Target Dummy", is_npc=True,
        str_=50, dex_=50, int_=50, hp=500,
        mana=500, stamina=200,
        skills={SKILLID_WRESTLING: 60, SKILLID_MEDITATION: 50},
        armor=ArmorSpec(name="Plate", ar=30),
    ),
    iterations=200,
    base_seed=42,
)

astral_result = run_scenario(astral_scenario, shard=shard)

# Astral damage drains mana/stamina, not HP — final_damage tracks ApplyRawDamage calls (0 for astral)
print(f"Successes: {astral_result.success_count}/{astral_result.iteration_count}")
print(f"Hit rate: {astral_result.ratios.hit_rate:.1%}")

# Inspect astral-specific metrics from a sample hit
sample_hits = [h for h in astral_result.raw_results if "astral_basedamage" in h.metrics]
if sample_hits:
    h = sample_hits[0]
    print(f"\nSample astral hit metrics:")
    print(f"  Base damage (after Spirit Speak scaling): {h.metrics['astral_basedamage']}")
    print(f"  Raw damage (after 50% reduction):        {h.metrics['astral_rawdamage']}")
    print(f"  Absorbed by astral armor:                {h.metrics['astral_absorbed']}")
    print(f"  Effective astral AR:                     {h.metrics['astral_ar']}")
    print(f"  Meditation triggered:                    {bool(h.metrics['astral_meditation_triggered'])}")

## Spell Casting Comparison (V3)

Direct spell casting — run a single **Fireball** scenario and compare against the weapon-hit results above. Spell casting uses `SpellScenario` and `run_spell_scenario()` instead of `Scenario` / `run_scenario()`.

In [ ]:
from omega.config.spells import Spell
from omega.simulation import SpellScenario, run_spell_scenario
from omega.reporting.plots import spell_comparison
from omega.reporting.tables import comparison_table

# Mage caster with high Magery + EvalInt for reliable casting
from omega.model.constants import SKILLID_MAGERY, SKILLID_EVALINT, SKILLID_MAGICRESISTANCE

mage = CombatantSpec(
    name="Mage",
    skills={SKILLID_MAGERY: 100, SKILLID_EVALINT: 100},
    str_=50, dex_=50, int_=120,
    class_levels={"IsMage": 5},
)

# Same NPC target as the weapon scenario above
spell_scenario = SpellScenario(
    caster=mage,
    target=CombatantSpec(
        name="Target Dummy", is_npc=True,
        str_=50, dex_=50, int_=50, hp=500,
        skills={SKILLID_WRESTLING: 60},
        armor=ArmorSpec(name="Plate", ar=30),
    ),
    spell_id=Spell.FIREBALL,
    iterations=200,
    base_seed=42,
    npc_mode=False,  # Full spell casting pipeline (TryToCast → CheckSkill → damage)
)

spell_result = run_spell_scenario(spell_scenario, shard=shard)

print(f"Spell: Fireball (Circle {spell_result.raw_results[0].circle})")
print(f"Fizzle rate:  {spell_result.ratios.fizzle_rate:.1%}")
print(f"Resist rate:  {spell_result.ratios.resist_rate:.1%}")
print(f"Cast rate:    {spell_result.ratios.hit_rate:.1%}")
print(f"Mean damage (overall):  {spell_result.damage_stats.mean:.2f}")
print(f"Mean damage (on cast):  {spell_result.damage_stats_on_cast.mean:.2f}")

In [ ]:
# Histogram of spell damage (fizzles show as 0 damage)
damage_histogram(spell_result, title="Fireball — 200 casts")

In [ ]:
# Side-by-side: weapon hit vs spell cast
cells = {"Warrior (weapon)": result, "Mage (Fireball)": spell_result}
rows = comparison_table(
    cells,
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95",
           "hit_rate", "fizzle_rate", "resist_rate",
           "effective_dps"],
)
HTML(format_table_html(rows))